In [1]:
# cargar el dataset mtcars
import pandas as pd
import statsmodels.api as sm
# Cargar el dataset mtcars
mtcars = sm.datasets.get_rdataset("mtcars").data
# Mostrar las primeras filas del dataset
mtcars.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
rownames,,,,,,,,,,,
Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [2]:
mtcars.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32 entries, Mazda RX4 to Volvo 142E
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   mpg     32 non-null     float64
 1   cyl     32 non-null     int64  
 2   disp    32 non-null     float64
 3   hp      32 non-null     int64  
 4   drat    32 non-null     float64
 5   wt      32 non-null     float64
 6   qsec    32 non-null     float64
 7   vs      32 non-null     int64  
 8   am      32 non-null     int64  
 9   gear    32 non-null     int64  
 10  carb    32 non-null     int64  
dtypes: float64(5), int64(6)
memory usage: 3.0+ KB


In [3]:
import pandas as pd
import numpy as np
from typing import List, Union, Optional
from typing_extensions import Annotated
from pydantic import BaseModel, Field

# Simulación de datos del dataset mtcars con estructura tipada
np.random.seed(42)

# Generación de muestra simulada de vehículos
n_vehiculos = 10
datos_simulados = {
    'modelo': [f'Auto_{i}' for i in range(n_vehiculos)],
    'mpg': np.random.uniform(10.4, 33.9, n_vehiculos).round(1),
    'cyl': np.random.choice([4, 6, 8], n_vehiculos),
    'hp': np.random.randint(52, 335, n_vehiculos),
    'wt': np.random.uniform(1.5, 5.4, n_vehiculos).round(2)
}

print("Datos simulados generados:")
for clave, valores in datos_simulados.items():
    print(f"  {clave}: {valores[:5]}...")


Datos simulados generados:
  modelo: ['Auto_0', 'Auto_1', 'Auto_2', 'Auto_3', 'Auto_4']...
  mpg: [19.2 32.7 27.6 24.5 14.1]...
  cyl: [6 4 6 6 6]...
  hp: [304 287 100 110 221]...
  wt: [2.99 5.33 3.32 4.85 4.15]...


In [5]:
cars = pd.DataFrame(datos_simulados)
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   modelo  10 non-null     object 
 1   mpg     10 non-null     float64
 2   cyl     10 non-null     int64  
 3   hp      10 non-null     int64  
 4   wt      10 non-null     float64
dtypes: float64(2), int64(2), object(1)
memory usage: 532.0+ bytes


In [33]:
# 1. Definición del tipo para matriz de covariables continuas
import numpy.typing as npt

# Anotación de arreglos multidimensionales con precisión especificada
MatrizDisenoCovariables = npt.NDArray[np.float64]

# 2. Definición del modelo de observación tipado
class EspecificacionCoche(BaseModel):
    """
    Modelo de validación para una observación individual del dataset mtcars.
    Cada campo representa una variable del diseño experimental.
    """
    # Identificador nominal del sujeto (puede estar perdido)
    modelo_vehiculo: Optional[str] = Field(
        default=None, 
        description="Identificador nominal del sujeto experimental"
    )
    
    # Caballos de fuerza: permite polimorfismo int/float
    caballos_fuerza_hp: int | float = Field(
        ..., 
        description="Medición de potencia bruta del motor",
        gt=0  # Restricción: debe ser positivo
    )
    
    # Vector de mediciones repetidas del cuarto de milla
    qsec_mediciones_repetidas: List[float] = Field(
        default_factory=list,
        description="Tiempos en cuarto de milla (medidas repetidas)"
    )
    
    # Variable indicadora con restricciones embebidas
    transmision_am_dummy: Annotated[int, Field(
        ge=0, le=1, 
        description="Variable dummy: 0 = Automático, 1 = Manual"
    )]

# 3. Función con firma tipada para ajuste de modelo
def ajustar_modelo_lineal(
    X: MatrizDisenoCovariables,
    y: npt.NDArray[np.float64],
    datos_completos: pd.DataFrame
) -> dict:
    """
    Función que exige matriz de diseño estrictamente numérica flotante.
    Calcula estimadores básicos de regresión lineal.
    """
    n, p = X.shape
    
    # Cálculo de estimadores por mínimos cuadrados ordinarios
# \( \hat{\beta} = (X'X)^{-1}X'y \)
    XtX_inv = np.linalg.inv(X.T @ X)
    beta_hat = XtX_inv @ (X.T @ y)
    
    # Residuales y estimación de varianza
    y_pred = X @ beta_hat
    residuales = y - y_pred
    sigma2_hat = np.sum(residuales**2) / (n - p)
    
    return {
        'coeficientes': beta_hat,
        'varianza_residual': sigma2_hat,
        'n_observaciones': n,
        'p_parametros': p
    }

# 4. Instanciación y validación de una observación
coche_valido = EspecificacionCoche(
    modelo_vehiculo="",
    caballos_fuerza_hp="12.4",
    qsec_mediciones_repetidas=[16.46, "12.5", 16.48],
    transmision_am_dummy=1
)

print(f"✅ Vehículo validado: {coche_valido.modelo_vehiculo}")
print(f"   HP: {coche_valido.caballos_fuerza_hp}")
print(f"   Transmisión: {'Manual' if coche_valido.transmision_am_dummy else 'Automático'}")

✅ Vehículo validado: 
   HP: 12.4
   Transmisión: Manual


In [1]:
import numpy as np
import pandas as pd

# Simulación de datos del dataset iris con valores anómalos intencionales
np.random.seed(42)

n_muestras = 15
especies = ['setosa', 'versicolor', 'virginica']

# Generación de datos mayormente válidos con algunos errores
datos_iris_simulados = []
for i in range(n_muestras):
    observacion = {
        'especie': np.random.choice(especies),
        'sepal_length': np.random.uniform(4.3, 7.9),
        'sepal_width': np.random.uniform(2.0, 4.4),
        'petal_length': np.random.uniform(1.0, 6.9),
        'petal_width': np.random.uniform(0.1, 2.5)
    }
    # Introducir errores deliberados en algunas observaciones
    if i == 3:  # Valor negativo imposible
        observacion['sepal_length'] = -5.1
    if i == 7:  # Valor excesivamente grande (error de escala)
        observacion['petal_length'] = 85.0
    if i == 11:  # Especie no válida
        observacion['especie'] = 'rosa'
        
    datos_iris_simulados.append(observacion)

print(f"Simuladas {n_muestras} observaciones con errores en índices 3, 7 y 11")

Simuladas 15 observaciones con errores en índices 3, 7 y 11


In [2]:
iris_data = pd.DataFrame(datos_iris_simulados)
iris_data.head(15)

,especie,sepal_length,sepal_width,petal_length,petal_width
0,virginica,7.167555,2.440243,5.600177,1.532440
1,versicolor,4.861580,2.139401,6.110439,1.542676
2,virginica,4.374104,4.327784,5.911412,0.609614
3,setosa,-5.100000,3.467968,1.041691,0.155350
4,virginica,6.502670,2.334785,2.723653,0.979268
5,versicolor,4.626183,3.484126,3.256526,2.459754
6,setosa,4.467221,3.458108,2.006092,0.256124
7,setosa,7.776275,3.940154,85.000000,0.334413
8,virginica,6.759749,3.463992,5.915850,0.516075
9,setosa,5.231608,3.590053,2.839095,1.348163


In [3]:
import time
import functools
import logging
from typing import ClassVar, List
from pydantic import BaseModel, field_validator, ValidationError

# 1. Decorador para cronometrar rutinas de validación masiva


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 1. Decorador mejorado
def cronometrar_bootstrap(func):
    """
    Decorador que mide el tiempo de ejecución de rutinas iterativas
    como validaciones masivas o remuestreos de Bootstrap.
    """
    @functools.wraps(func)
    def envoltorio_tiempo(*args, **kwargs):
        tiempo_inicio = time.perf_counter()
        try:
            resultado = func(*args, **kwargs)
            return resultado
        finally:
            tiempo_fin = time.perf_counter()
            duracion = tiempo_fin - tiempo_inicio
            # Usamos logger en lugar de print
            logger.info(f"La rutina '{func.__name__}' ejecutó en {duracion:.5f} segundos.")
    return envoltorio_tiempo

# 2. Modelo de validación para observaciones del dataset iris
class ObservacionIris(BaseModel):
    """
    Modelo estadístico para validación de observaciones florales.
    Incluye validadores para garantizar axiomas biológicos.
    """
    # Variable dependiente (factor categórico)
    especie_taxonomica: str
    
    # Vector de variables predictoras continuas
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float
    
    # Constantes de clase: niveles permitidos del factor
    ESPECIES_PERMITIDAS: ClassVar[List[str]] = ["setosa", "versicolor", "virginica"]
    
    @field_validator('especie_taxonomica')
    @classmethod
    def validar_factor_categorico(cls, valor: str) -> str:
        """
        Garantiza que el factor categórico solo contenga niveles
        poblacionales conocidos. Previene 'niveles fantasma' que
        destruirían los grados de libertad en análisis posteriores.
        """
        valor_limpio = valor.strip().lower()
        if valor_limpio not in cls.ESPECIES_PERMITIDAS:
            raise ValueError(
                f"Anomalía Taxonómica. Especie no reconocida: '{valor}'. "
                f"El hiperespacio poblacional permite: {cls.ESPECIES_PERMITIDAS}"
            )
        return valor_limpio
    
    @field_validator('sepal_length', 'sepal_width', 'petal_length', 'petal_width', mode='after')
    @classmethod
    def auditar_axiomas_biologicos(cls, valor: float) -> float:
        """
        Comprueba axiomas del dominio del proceso generador de datos:
        las medidas morfológicas deben ser estrictamente positivas.
        """
        # Restricción teórica: valores positivos
        if valor <= 0:
            raise ValueError(
                "Violación Axiomática: Las mediciones biológicas deben "
                "residir en la región topológica \(0, +\infty\)."
            )

        # Heurística de control de calidad: detectar errores de escala
        if valor > 50.0:
            raise ValueError(
                f"Medición atípica sospechosa ({valor} cm); posible error "
                "de transcripción de milímetros a centímetros."
            )
        return valor

# 3. Simulación de ingesta muestral masiva con manejo de errores
@cronometrar_bootstrap
def simular_ingesta_bootstrap(lista_datos: list) -> dict:
    """
    Procesa una lista de observaciones simulando un pipeline de ingesta
    con validación automatizada y registro de exclusiones.
    """
    observaciones_validas = []
    errores_detectados = []
    total_registros = len(lista_datos)
    
    for i, dato in enumerate(lista_datos):
        try:
            medicion = ObservacionIris(**dato)
            observaciones_validas.append(medicion)
        except ValidationError as error:
            errores_detectados.append({
                'indice': i,
                'dato_original': dato,
                'error': str(error)
            })
            logger.warning(f"⚠️ [Rechazo #{len(errores_detectados)}] Observación índice {i}: {error.errors()[0]['msg'][:50]}...")
    
    return {
        'validas': observaciones_validas,
        'rechazadas': errores_detectados,
        'tasa_exclusion': len(errores_detectados) / len(lista_datos) * 100
    }

# 4. Ejecución de la simulación
if __name__ == "__main__":
    try:
        resultado = simular_ingesta_bootstrap(datos_iris_simulados)
        logger.info(f"\n📊 Resumen de Validación:")
        logger.info(f"   Aceptadas: {len(resultado['validas'])}")
        logger.info(f"   Rechazadas: {len(resultado['rechazadas'])}")
        logger.info(f"   Tasa de exclusión: {resultado['tasa_exclusion']:.1f}%")
    except Exception as e:
        logger.error(f"Fallo crítico en el pipeline: {e}")

2026-03-09 21:45:29,795 - WARNING - ⚠️ [Rechazo #1] Observación índice 0: Field required...
2026-03-09 21:45:29,796 - WARNING - ⚠️ [Rechazo #2] Observación índice 1: Field required...
2026-03-09 21:45:29,796 - WARNING - ⚠️ [Rechazo #3] Observación índice 2: Field required...
2026-03-09 21:45:29,796 - WARNING - ⚠️ [Rechazo #4] Observación índice 3: Field required...
2026-03-09 21:45:29,796 - WARNING - ⚠️ [Rechazo #5] Observación índice 4: Field required...
2026-03-09 21:45:29,796 - WARNING - ⚠️ [Rechazo #6] Observación índice 5: Field required...
2026-03-09 21:45:29,797 - WARNING - ⚠️ [Rechazo #7] Observación índice 6: Field required...
2026-03-09 21:45:29,797 - WARNING - ⚠️ [Rechazo #8] Observación índice 7: Field required...
2026-03-09 21:45:29,797 - WARNING - ⚠️ [Rechazo #9] Observación índice 8: Field required...
2026-03-09 21:45:29,797 - WARNING - ⚠️ [Rechazo #10] Observación índice 9: Field required...
2026-03-09 21:45:29,798 - WARNING - ⚠️ [Rechazo #11] Observación índice 10: Fie

In [4]:
resultado

{'validas': [],
 'rechazadas': [{'indice': 0,
   'dato_original': {'especie': np.str_('virginica'),
    'sepal_length': 7.167554752696838,
    'sepal_width': 2.4402434956787933,
    'petal_length': 5.600176901609339,
    'petal_width': 1.5324403790715688},
   'error': "1 validation error for ObservacionIris\nespecie_taxonomica\n  Field required [type=missing, input_value={'especie': np.str_('virg...th': 1.5324403790715688}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.11/v/missing"},
  {'indice': 1,
   'dato_original': {'especie': np.str_('versicolor'),
    'sepal_length': 4.861580273210329,
    'sepal_width': 2.139400669203679,
    'petal_length': 6.110439260072118,
    'petal_width': 1.542676028183701},
   'error': "1 validation error for ObservacionIris\nespecie_taxonomica\n  Field required [type=missing, input_value={'especie': np.str_('vers...dth': 1.542676028183701}, input_type=dict]\n    For further information visit https://errors.pydantic.d

In [74]:
resultado_df = pd.DataFrame(resultado['rechazadas'])
resultado_df[['dato_original']].iloc[1].tolist()

[{'especie': np.str_('versicolor'),
  'sepal_length': 4.861580273210329,
  'sepal_width': 2.139400669203679,
  'petal_length': 6.110439260072118,
  'petal_width': 1.542676028183701}]